In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2
Configuration: {'general': {'run_name': 'experiment_with_05_classes', 'seed': 42}, 'dataset': {'split_type': 'test', 'n_classes': 25, 'n_samples_per_class': 100}, 'paths': {'data_exploration_dir': 'output/experiment_with_05_classes/data_exploration', 'embeddings_dir': 'output/experiment_with_05_classes/embeddings', 'models_dir': 'output/experiment_with_05_classes/models', 'results_dir': 'output/experiment_with_05_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_N_CLASSES: 25
  DATASET_N_SAMPLES_PER_CLASS: 100
  DATASET_SPLIT_TYPE: test

# Build Embeddings
This notebook generates embeddings for both SBERT (Sequence encoder) and OpenAI models and stores the indices inside the experiment folder.

In [2]:

import pathlib, json, numpy as np, faiss
from tqdm import tqdm
from src.datasets.dataset import get_dataset
from src.rag.vector_store import VectorStore
from src.rag import _EMBEDDINGS_DIR, _SBERT_DIR, _OPENAI_DIR
from src.embeddings.openai_embedder import OpenAIEmbedder
from sentence_transformers import SentenceTransformer

# ---- Parameters ----
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OPENAI_MODEL = "text-embedding-3-small"

print('Embeddings root:', _EMBEDDINGS_DIR)
_SBERT_DIR.mkdir(parents=True, exist_ok=True)
_OPENAI_DIR.mkdir(parents=True, exist_ok=True)


INFO | Loading faiss with AVX2 support.
INFO | Successfully loaded faiss with AVX2 support.
INFO | Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.
/home/marcmaceira/projects/reuters-rag-classifier_clean_v2/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embeddings root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_05_classes/embeddings


In [3]:

# Load dataset
# X_train, y_train, _, _, _ = get_dataset(split_type="standard", n_classes=N_CLASSES)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
    n_samples_per_class=N_SAMPLES_PER_CLASS,
)

print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")


INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: test
INFO |   - Number of classes: 25
INFO |   - Samples per class: 100
INFO |   - Random seed: None
INFO | Loading small test dataset with 100 samples per class across 25 classes
INFO | Selected classes: earn, acq, crude, interest, money-fx, trade, grain, corn, dlr, money-supply, ship, coffee, sugar, gold, bop, gnp, cpi, cocoa, carcass, oilseed, copper, alum, reserves, jobs, barley
INFO |   - Class 'earn': 70 train, 30 test
INFO |   - Class 'acq': 70 train, 30 test
INFO |   - Class 'crude': 70 train, 30 test
INFO |   - Class 'interest': 70 train, 30 test
INFO |   - Class 'money-fx': 70 train, 30 test
INFO |   - Class 'trade': 70 train, 30 test
INFO |   - Class 'grain': 70 train, 30 test
INFO |   - Class 'corn': 70 train, 30 test
INFO |   - Class 'dlr': 70 train, 30 test
INFO |   - Class 'money-supply': 70 train, 30 test
INFO |   - Class 'ship': 70 train, 30 test
INFO |   - Class 'coffee': 70 train, 30 test
INFO 

Loaded 1511 training documents with 25 classes


In [4]:

# ---- SBERT embeddings ----
sbert = SentenceTransformer(SBERT_MODEL)
vectors = sbert.encode(X_train, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype('float32')

meta = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, vectors)):
    meta.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

faiss_path = _SBERT_DIR / "index.faiss"
meta_path  = _SBERT_DIR / "meta.jsonl"
VectorStore.build(vectors, meta, vectors.shape[1], faiss_path, meta_path)
print("✅ SBERT index saved at", faiss_path)


INFO | Use pytorch device_name: cpu
INFO | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
Batches: 100%|██████████| 24/24 [00:37<00:00,  1.55s/it]
INFO | Building FAISS index with 1511 documents of dimension 384
INFO | Creating IndexFlatIP...
INFO | Normalizing embeddings...
INFO | Adding embeddings to index...
INFO | Added embeddings to index in 0.00 seconds
INFO | Writing index to /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_05_classes/embeddings/sbert/index.faiss...
INFO | Wrote index in 0.00 seconds
INFO | Writing metadata to /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_05_classes/embeddings/sbert/meta.jsonl...
INFO | Wrote metadata in 0.51 seconds
INFO | Index built in 0.52 seconds


✅ SBERT index saved at /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_05_classes/embeddings/sbert/index.faiss


In [5]:

# ---- OpenAI embeddings ----
# Requires OPENAI_API_KEY env var
openai_embedder = OpenAIEmbedder(model=OPENAI_MODEL, batch_size=50)
openai_vecs = openai_embedder.encode(X_train)
openai_vecs = np.array(openai_vecs, dtype='float32')

meta_openai = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, openai_vecs)):
    meta_openai.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

openai_faiss = _OPENAI_DIR / "index.faiss"
openai_meta  = _OPENAI_DIR / "meta.jsonl"
VectorStore.build(openai_vecs, meta_openai, openai_vecs.shape[1], openai_faiss, openai_meta)
print("✅ OpenAI index saved at", openai_faiss)


INFO | Starting OpenAI embedding generation for 1511 texts with model text-embedding-3-small
INFO | Processing embedding batch 1 with 50 texts
INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO | Processed batch 1/1 (50 texts) in 1.52s
INFO | Batch 1 completed in 1.53 seconds
INFO | Average time per text in batch: 0.0306 seconds
INFO | Adding delay of 0.43s before next batch
INFO | Processing embedding batch 2 with 50 texts
INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO | Processed batch 1/1 (50 texts) in 1.30s
INFO | Batch 2 completed in 1.30 seconds
INFO | Average time per text in batch: 0.0261 seconds
INFO | Adding delay of 0.36s before next batch
INFO | Processing embedding batch 3 with 50 texts
INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO | Processed batch 1/1 (50 texts) in 1.39s
INFO | Batch 3 completed in 1.40 seconds
INFO | Average time per text in batch: 0.0281 se

✅ OpenAI index saved at /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_05_classes/embeddings/openai/index.faiss
